In [1]:
# Import required libraries
import pandas as pd
import numpy as np

from sklearn.ensemble import IsolationForest

In [29]:
# Load feature engineered dataset
df = pd.read_csv("C:/Users/BIT/Desktop/AIML ASSIGNMENT/data/feature_engineered_data.csv")

# Convert Date column to datetime
df['Date'] = pd.to_datetime(df['Date'])

# Sort data (good practice)
df = df.sort_values(by='Date')

df.head()

,Date,Person_ID,Age,Gender,Temperature_C,Activity_Level,Water_Consumed_Liters,Consumption_scaled,Day,Month,Year,Day_of_Week,Rolling_Mean_3,Rolling_Std_3,Lag_1,City_Islamabad,City_Karachi,City_Lahore,City_Multan,City_Rawalpindi
0,2025-01-01,P0061,41.0,1.0,17.7,0,1.76,0.160279,1,1,2025,2,2.330000,0.538424,2.83,False,True,False,False,False
66,2025-01-01,P0044,22.0,0.0,37.1,0,2.67,0.477352,1,1,2025,2,2.923333,0.271539,2.89,True,False,False,False,False
65,2025-01-01,P0065,31.0,0.0,28.6,1,2.89,0.554007,1,1,2025,2,2.576667,0.835304,3.21,False,False,True,False,False
64,2025-01-01,P0099,60.0,1.0,38.0,1,3.21,0.665505,1,1,2025,2,2.680000,0.909340,1.63,False,False,False,False,False
63,2025-01-01,P0046,36.0,0.0,16.9,0,1.63,0.114983,1,1,2025,2,2.680000,0.909340,3.20,False,False,False,False,False


In [30]:
# Select relevant features for anomaly detection
features = df[['Water_Consumed_Liters', 'Temperature_C']]

# Initialize Isolation Forest model
model = IsolationForest(contamination=0.05, random_state=42)

# Fit model and predict anomalies
df['Anomaly'] = model.fit_predict(features)

# Anomaly: -1 → abnormal, 1 → normal

In [31]:
# Create Time Index
df['Time_Index'] = np.arange(len(df))

# Define X and y
X = df[['Time_Index']]
y = df['Water_Consumed_Liters']

# Train model
from sklearn.linear_model import LinearRegression

model = LinearRegression()
model.fit(X, y)

# Predict future 7 days
future_days = 7
last_index = df['Time_Index'].iloc[-1]

future_index = np.arange(last_index + 1, last_index + 1 + future_days).reshape(-1, 1)

future_predictions = model.predict(future_index)

c:\Users\BIT\Desktop\AIML ASSIGNMENT\.venv\Lib\site-packages\sklearn\utils\validation.py:2691: UserWarning: X does not have valid feature names, but LinearRegression was fitted with feature names
  warnings.warn(


In [32]:
#Anamoly-Based Detection
# Create alert column based on anomaly detection
df['Anomaly_Alert'] = df['Anomaly'].apply(
    lambda x: "⚠️ Abnormal Usage Detected" if x == -1 else "Normal"
)

# Filter anomaly rows
anomaly_alerts = df[df['Anomaly'] == -1]

print("=== Anomaly Alerts ===\n")
print(anomaly_alerts[['Date', 'Water_Consumed_Liters', 'Anomaly_Alert']])

=== Anomaly Alerts ===

            Date  Water_Consumed_Liters               Anomaly_Alert
63    2025-01-01                   1.63  ⚠️ Abnormal Usage Detected
56    2025-01-01                   1.56  ⚠️ Abnormal Usage Detected
49    2025-01-01                   2.76  ⚠️ Abnormal Usage Detected
11    2025-01-01                   1.76  ⚠️ Abnormal Usage Detected
147   2025-01-02                   1.60  ⚠️ Abnormal Usage Detected
...          ...                    ...                         ...
32884 2025-12-31                   1.61  ⚠️ Abnormal Usage Detected
32837 2025-12-31                   1.57  ⚠️ Abnormal Usage Detected
32851 2025-12-31                   2.64  ⚠️ Abnormal Usage Detected
32836 2025-12-31                   3.92  ⚠️ Abnormal Usage Detected
32843 2025-12-31                   1.40  ⚠️ Abnormal Usage Detected

[1645 rows x 3 columns]


In [33]:
#Threshold-Based Alerts
# Define threshold using statistical approach
threshold = df['Water_Consumed_Liters'].mean() + df['Water_Consumed_Liters'].std()

# Generate alerts based on threshold
df['Threshold_Alert'] = df['Water_Consumed_Liters'].apply(
    lambda x: "⚠️ High Consumption" if x > threshold else "Normal"
)

# Filter high consumption rows
threshold_alerts = df[df['Water_Consumed_Liters'] > threshold]

print("\n=== Threshold Alerts ===\n")
print(threshold_alerts[['Date', 'Water_Consumed_Liters', 'Threshold_Alert']])


=== Threshold Alerts ===

            Date  Water_Consumed_Liters      Threshold_Alert
59    2025-01-01                   3.70  ⚠️ High Consumption
58    2025-01-01                   3.43  ⚠️ High Consumption
89    2025-01-01                   3.29  ⚠️ High Consumption
17    2025-01-01                   3.51  ⚠️ High Consumption
14    2025-01-01                   3.54  ⚠️ High Consumption
...          ...                    ...                  ...
32848 2025-12-31                   3.66  ⚠️ High Consumption
32836 2025-12-31                   3.92  ⚠️ High Consumption
32847 2025-12-31                   3.78  ⚠️ High Consumption
32842 2025-12-31                   3.37  ⚠️ High Consumption
32841 2025-12-31                   3.54  ⚠️ High Consumption

[5736 rows x 3 columns]


In [34]:
#Forecast-Based Alerts
# Example fallback (remove if already defined)
# future_predictions = [2.5, 3.1, 4.0, 2.8, 3.5, 4.2, 3.0]

print("\n=== Forecast Alerts (Next 7 Days) ===\n")

forecast_threshold = df['Water_Consumed_Liters'].mean()

for i, value in enumerate(future_predictions):
    if value > forecast_threshold:
        print(f"⚠️ Day {i+1}: High predicted consumption ({value:.2f})")
    else:
        print(f"Day {i+1}: Normal consumption ({value:.2f})")


=== Forecast Alerts (Next 7 Days) ===

Day 1: Normal consumption (2.71)
Day 2: Normal consumption (2.71)
Day 3: Normal consumption (2.71)
Day 4: Normal consumption (2.71)
Day 5: Normal consumption (2.71)
Day 6: Normal consumption (2.71)
Day 7: Normal consumption (2.71)


In [35]:
# Combine all alerts into a single decision system

def generate_final_alert(row):
    if row['Anomaly'] == -1:
        return "⚠️ Anomaly Detected"
    elif row['Water_Consumed_Liters'] > threshold:
        return "⚠️ High Consumption"
    else:
        return "Normal"

# Apply function
df['Final_Alert'] = df.apply(generate_final_alert, axis=1)

# Display sample results
print("\n=== Final Alert Summary ===\n")
print(df[['Date', 'Water_Consumed_Liters', 'Final_Alert']].head(20))


=== Final Alert Summary ===

         Date  Water_Consumed_Liters          Final_Alert
0  2025-01-01               1.760000               Normal
66 2025-01-01               2.670000               Normal
65 2025-01-01               2.890000               Normal
64 2025-01-01               3.210000               Normal
63 2025-01-01               1.630000  ⚠️ Anomaly Detected
62 2025-01-01               3.200000               Normal
61 2025-01-01               3.210000               Normal
60 2025-01-01               2.240000               Normal
59 2025-01-01               3.700000  ⚠️ High Consumption
58 2025-01-01               3.430000  ⚠️ High Consumption
57 2025-01-01               1.960000               Normal
56 2025-01-01               1.560000  ⚠️ Anomaly Detected
55 2025-01-01               2.510000               Normal
54 2025-01-01               2.722212               Normal
53 2025-01-01               2.100000               Normal
52 2025-01-01               2.270000      

In [37]:
# Save dataset with alerts
df.to_csv("C:/Users/BIT/Desktop/AIML ASSIGNMENT/data/alert_results.csv", index=False)

print("Alert results saved successfully!")

Alert results saved successfully!
